# Next-frame prediction with a Linear Model

In [4]:
import os
import numpy as np
from sklearn.linear_model import Ridge
from dotenv import load_dotenv

from impulse import (
    ReplayDataset,
    PreprocessingPipeline,
    FeatureSelector,
    PhysicalNormalizer,
    split_replay_ids
)
from impulse.preprocessing import deserialize_boundaries

load_dotenv()

True

In [8]:
impulse_path = os.getenv('IMPULSE_PATH')
db_path = os.getenv('DB_PATH')
project_root = os.getenv('PROJECT_ROOT')
cache_dir = project_root+'replay-cache/'

dataset = ReplayDataset(db_path=db_path, cache_dir=cache_dir)

In [11]:
len(dataset)

7282

In [13]:
sample_ids = dataset.sample_replay_ids(n=200, seed=51)
len(sample_ids)

200

In [14]:
train_ids, val_ids, test_ids = split_replay_ids(sample_ids)

In [17]:
pipeline = PreprocessingPipeline([
    FeatureSelector.from_preset('physics'),
    PhysicalNormalizer()
])

In [18]:
X_train, Y_train = [], []

for replay in dataset.iter_ids(train_ids):
    segment_boundaries = deserialize_boundaries(dataset.replay_info[replay.replay_id]['segment_boundaries'])
    processed = pipeline(replay.frames).to_numpy(np.float32)
    for start, end in segment_boundaries:
        segment = processed[start:end]
        X_train.append(segment[:-1])
        Y_train.append(segment[1:])

X_train = np.concatenate(X_train)
Y_train = np.concatenate(Y_train)

In [19]:
X_val, Y_val = [], []
for replay in dataset.iter_ids(val_ids):
    boundaries = deserialize_boundaries(dataset.replay_info[replay.replay_id]['segment_boundaries'])
    processed = pipeline(replay.frames).to_numpy(np.float32)
    for start, end in boundaries:
        segment = processed[start:end]
        X_val.append(segment[:-1])
        Y_val.append(segment[1:])

X_val = np.concatenate(X_val)
Y_val = np.concatenate(Y_val)

In [20]:
trivial_mse = ((Y_val - X_val) ** 2).mean(axis=0)

In [ ]:
model = Ridge(alpha=1.0)
model.fit(X_train, Y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [22]:
Y_pred = model.predict(X_val)
linear_mse = ((Y_val - Y_pred) ** 2).mean(axis=0)

In [23]:
feature_names = pipeline.feature_columns
for i, name in enumerate(feature_names):
    improvement = (1 - linear_mse[i] / trivial_mse[i]) * 100
    print(f"{name:40s}  trivial={trivial_mse[i]:.6f}  linear={linear_mse[i]:.6f}  improvement={improvement:.1f}%")

Ball - position x                         trivial=0.000117  linear=0.000032  improvement=72.2%
Ball - position y                         trivial=0.000127  linear=0.000052  improvement=58.8%
Ball - position z                         trivial=0.000129  linear=0.000032  improvement=75.3%
Ball - linear velocity x                  trivial=0.001821  linear=0.001780  improvement=2.3%
Ball - linear velocity y                  trivial=0.002830  linear=0.002750  improvement=2.8%
Ball - linear velocity z                  trivial=0.001063  linear=0.001016  improvement=4.4%
Ball - angular velocity x                 trivial=129.300964  linear=127.582985  improvement=1.3%
Ball - angular velocity y                 trivial=102.708252  linear=101.411827  improvement=1.3%
Ball - angular velocity z                 trivial=139.762238  linear=138.087585  improvement=1.2%
Ball - quaternion x                       trivial=0.039430  linear=0.037523  improvement=4.8%
Ball - quaternion y                       tri